# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/content_refresh_anonymized.csv")
df.shape


(30000, 44)

In [8]:
df.head()
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

staleness_check

,staleness_bucket,n,median_impressions,median_clicks,median_ctr
0,0-30,20480,470.0,1.0,0.04
1,31-90,175,510.0,0.0,0.00
2,91-180,9171,1692.0,2.0,0.10
3,181-365,169,16.0,0.0,0.00
4,365+,5,2.0,0.0,0.00


In [12]:
df["volume_bucket"] = pd.qcut(
    df["search_volume"],
    q=4,
    duplicates="drop"
)

volume_check = (
    df.groupby("volume_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

volume_check

,volume_bucket,n,median_impressions,median_clicks,median_ctr
0,"(-0.001, 10.0]",18392,929.0,1.0,0.09
1,"(10.0, 20.0]",2290,1006.5,1.0,0.10
2,"(20.0, 74000.0]",6850,842.5,1.0,0.05


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# Create percentile scores
df["staleness_score"] = df["days_since_last_update"].rank(pct=True)

df["volume_score"] = df["search_volume"].rank(pct=True)

# Baseline score
df["baseline_score"] = (
    0.6 * df["staleness_score"] +
    0.4 * df["volume_score"]
)

# One reason code
df["reason_code"] = np.where(
    (df["staleness_score"] >= 0.7) &
    (df["volume_score"] >= 0.7),
    "STALE_HIGH_DEMAND",
    "LOWER_PRIORITY"
)

# One action label
df["action"] = np.where(
    df["reason_code"] == "STALE_HIGH_DEMAND",
    "Refresh Content",
    "Monitor"
)

# Rank everything
baseline = df.sort_values(
    "baseline_score",
    ascending=False
).copy()

baseline["rank"] = range(1, len(baseline) + 1)

# Select output columns
output = baseline[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

# Write required CSV
output = baseline[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

# Create the CSV file
output.to_csv(
    "baseline_action_score.csv",
    index=False
)

print("CSV file created successfully.")
display(output.head(10))

CSV file created successfully.


,rank,content_id,baseline_score,reason_code,action
1659,1,content_bbca724138f2,0.992793,STALE_HIGH_DEMAND,Refresh Content
12565,2,content_a31e10779c01,0.992546,STALE_HIGH_DEMAND,Refresh Content
15947,3,content_40e140ba2934,0.987274,STALE_HIGH_DEMAND,Refresh Content
23619,4,content_24abafed9707,0.982901,STALE_HIGH_DEMAND,Refresh Content
15789,5,content_23e958c54c78,0.979311,STALE_HIGH_DEMAND,Refresh Content
3968,6,content_29ec1008c834,0.974471,STALE_HIGH_DEMAND,Refresh Content
9417,7,content_c3dd69918c8c,0.974471,STALE_HIGH_DEMAND,Refresh Content
11722,8,content_6efb8fa48ebe,0.967882,STALE_HIGH_DEMAND,Refresh Content
2484,9,content_0cc405838fc5,0.967332,STALE_HIGH_DEMAND,Refresh Content
10601,10,content_17e6b2ba4b08,0.963518,STALE_HIGH_DEMAND,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
top20 = baseline.head(20).copy()

display(top20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,staleness_bucket,volume_bucket,staleness_score,volume_score,baseline_score,reason_code,action,rank
1659,content_bbca724138f2,client_6208ef0f77,1600.0,0.43,MEDIUM,3.76,keyword article,transactional,5614.0,37325.0,...,down,-100.0,181-365,"(20.0, 74000.0]",0.998800,0.983783,0.992793,STALE_HIGH_DEMAND,Refresh Content,1
12565,content_a31e10779c01,client_e29c9c180c,3600.0,0.06,LOW,0.26,keyword article,informational,4949.0,32888.0,...,flat,NaN,91-180,"(20.0, 74000.0]",0.992650,0.992391,0.992546,STALE_HIGH_DEMAND,Refresh Content,2
15947,content_40e140ba2934,client_8722616204,720.0,0.29,LOW,0.78,keyword article,transactional,3306.0,21228.0,...,flat,NaN,181-365,"(20.0, 74000.0]",0.998633,0.970235,0.987274,STALE_HIGH_DEMAND,Refresh Content,3
23619,content_24abafed9707,client_8722616204,480.0,0.00,LOW,0.00,keyword article,transactional,3246.0,21393.0,...,down,-100.0,181-365,"(20.0, 74000.0]",0.998633,0.959302,0.982901,STALE_HIGH_DEMAND,Refresh Content,4
15789,content_23e958c54c78,client_e29c9c180c,480.0,0.02,LOW,0.05,keyword article,informational,4832.0,31855.0,...,flat,NaN,91-180,"(20.0, 74000.0]",0.992650,0.959302,0.979311,STALE_HIGH_DEMAND,Refresh Content,5
3968,content_29ec1008c834,client_9f14025af0,320.0,0.00,LOW,0.00,keyword article,informational,1067.0,6851.0,...,flat,NaN,91-180,"(20.0, 74000.0]",0.993567,0.945827,0.974471,STALE_HIGH_DEMAND,Refresh Content,6
9417,content_c3dd69918c8c,client_9f14025af0,320.0,0.30,LOW,5.45,keyword article,informational,1048.0,7007.0,...,down,-100.0,91-180,"(20.0, 74000.0]",0.993567,0.945827,0.974471,STALE_HIGH_DEMAND,Refresh Content,7
11722,content_6efb8fa48ebe,client_9f14025af0,210.0,0.02,LOW,0.00,keyword article,informational,972.0,6259.0,...,down,-100.0,91-180,"(20.0, 74000.0]",0.993567,0.929355,0.967882,STALE_HIGH_DEMAND,Refresh Content,8
2484,content_0cc405838fc5,client_e29c9c180c,210.0,0.17,LOW,0.17,keyword article,informational,5117.0,34122.0,...,new,NaN,91-180,"(20.0, 74000.0]",0.992650,0.929355,0.967332,STALE_HIGH_DEMAND,Refresh Content,9
10601,content_17e6b2ba4b08,client_e29c9c180c,170.0,0.02,LOW,0.00,keyword article,transactional,5071.0,33895.0,...,down,-100.0,91-180,"(20.0, 74000.0]",0.992650,0.919821,0.963518,STALE_HIGH_DEMAND,Refresh Content,10


In [24]:
top20_review = top20[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["why_its_there"] = ""
top20_review["what_would_make_it_wrong"] = ""

display(top20_review)


,rank,content_id,baseline_score,reason_code,action,why_its_there,what_would_make_it_wrong
1659,1,content_bbca724138f2,0.992793,STALE_HIGH_DEMAND,Refresh Content,,
12565,2,content_a31e10779c01,0.992546,STALE_HIGH_DEMAND,Refresh Content,,
15947,3,content_40e140ba2934,0.987274,STALE_HIGH_DEMAND,Refresh Content,,
23619,4,content_24abafed9707,0.982901,STALE_HIGH_DEMAND,Refresh Content,,
15789,5,content_23e958c54c78,0.979311,STALE_HIGH_DEMAND,Refresh Content,,
3968,6,content_29ec1008c834,0.974471,STALE_HIGH_DEMAND,Refresh Content,,
9417,7,content_c3dd69918c8c,0.974471,STALE_HIGH_DEMAND,Refresh Content,,
11722,8,content_6efb8fa48ebe,0.967882,STALE_HIGH_DEMAND,Refresh Content,,
2484,9,content_0cc405838fc5,0.967332,STALE_HIGH_DEMAND,Refresh Content,,
10601,10,content_17e6b2ba4b08,0.963518,STALE_HIGH_DEMAND,Refresh Content,,


In [29]:
top20_review["why_its_there"] = np.where(
    (top20["staleness_score"] >= 0.70) &
    (top20["volume_score"] >= 0.70),
    "High staleness score combined with high search-demand score.",
    "High combined baseline score from the staleness and search-demand signals."
)
display(top20_review)

,rank,content_id,baseline_score,reason_code,action,why_its_there,what_would_make_it_wrong
1659,1,content_bbca724138f2,0.992793,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
12565,2,content_a31e10779c01,0.992546,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
15947,3,content_40e140ba2934,0.987274,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
23619,4,content_24abafed9707,0.982901,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
15789,5,content_23e958c54c78,0.979311,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
3968,6,content_29ec1008c834,0.974471,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
9417,7,content_c3dd69918c8c,0.974471,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
11722,8,content_6efb8fa48ebe,0.967882,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
2484,9,content_0cc405838fc5,0.967332,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,
10601,10,content_17e6b2ba4b08,0.963518,STALE_HIGH_DEMAND,Refresh Content,High staleness score combined with high search...,


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [28]:
# Inspect the Top-20 picks for potentially weak recommendations

top20_diagnostic = baseline.head(20)[
    [
        "rank",
        "content_id",
        "baseline_score",
        "days_since_last_update",
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "reason_code",
        "action"
    ]
].copy()

display(top20_diagnostic)

,rank,content_id,baseline_score,days_since_last_update,search_volume,impressions_90d,clicks_90d,ctr,avg_position,reason_code,action
1659,1,content_bbca724138f2,0.992793,236,1600.0,75,0,0.0,12.1,STALE_HIGH_DEMAND,Refresh Content
12565,2,content_a31e10779c01,0.992546,144,3600.0,1,0,0.0,2.0,STALE_HIGH_DEMAND,Refresh Content
15947,3,content_40e140ba2934,0.987274,231,720.0,2,0,0.0,4.5,STALE_HIGH_DEMAND,Refresh Content
23619,4,content_24abafed9707,0.982901,231,480.0,4,0,0.0,1.3,STALE_HIGH_DEMAND,Refresh Content
15789,5,content_23e958c54c78,0.979311,144,480.0,36,0,0.0,85.3,STALE_HIGH_DEMAND,Refresh Content
3968,6,content_29ec1008c834,0.974471,151,320.0,2,0,0.0,40.0,STALE_HIGH_DEMAND,Refresh Content
9417,7,content_c3dd69918c8c,0.974471,151,320.0,159,0,0.0,49.3,STALE_HIGH_DEMAND,Refresh Content
11722,8,content_6efb8fa48ebe,0.967882,151,210.0,3,0,0.0,68.3,STALE_HIGH_DEMAND,Refresh Content
2484,9,content_0cc405838fc5,0.967332,144,210.0,1,0,0.0,0.0,STALE_HIGH_DEMAND,Refresh Content
10601,10,content_17e6b2ba4b08,0.963518,144,170.0,4,0,0.0,4.8,STALE_HIGH_DEMAND,Refresh Content


## Which picks look wrong and why?

Several Top-20 picks look questionable rather than clearly wrong.

- **Rank 2:** High search volume and staleness produce a high score, but the page has only 1 impression and 0 clicks in the 90-day window. Search volume may therefore overstate the actual opportunity.
- **Rank 5:** The page is stale and has moderate search volume, but its average position is 85.3 with 0 clicks. A content refresh alone may not be sufficient.
- **Rank 9:** The page has a high baseline score despite only 1 impression and 0 clicks, suggesting that the rule may overvalue search volume.
- **Rank 11:** The page has high staleness but only 1 impression and 0 clicks, making it a weak immediate-refresh candidate.
- **Rank 16:** The score is driven by the two baseline signals, but the page has only 1 impression and 0 clicks.
- **Rank 19:** The page has low observed exposure (2 impressions and 0 clicks), so its high ranking may not represent a strong actionable opportunity.

These examples show a limitation of the baseline: it uses staleness and search demand but does not account for actual search exposure strongly enough. Therefore, some high-ranked pages may not be the best practical refresh opportunities.


## Leakage and Future-Window Check

The baseline score uses only:

- `days_since_last_update`
- `search_volume`

The columns `trend_direction` and `trend_pct` were not used as scoring features because they represent outcome/trend information that could introduce leakage.

No future-window or label-derived feature was used in the baseline score.

The 90-day performance fields were used only to inspect and validate the signals, not to calculate the final baseline score.

No product/flag field was used as a scoring feature.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.